# Issue #83 A100 Speculative-Rollout Roadmap and Validation Harness

**Hardware assumption:** validation target is an 8x NVIDIA A100 node. A100 is an Ampere target; this notebook avoids H100/SM90-specific assumptions.

**Artifact status:** planning/validation harness, not a benchmark result page.

**Explicit non-claims:**

- No measured speculative decoding speedup.
- No completed vLLM PagedAttention KV integration.
- No new CUDA kernel implementation.
- No real 8xA100 result unless a later run fills the schema with measured data.
- No current `vllm-rollout-prefix-cache` profiler workload is claimed.

This contribution is a reproducible validation path for shared-prefix, multi-candidate rollout around the current vLLM sampler and rollout executor anchors. Draft/target speculative decoding and acceptance accounting are future/planned instrumentation unless implemented and measured later.

In [ ]:
# Environment probe: dependency-light and safe on CPU-only machines.
import os
import platform
import sys
from datetime import datetime, timezone

probe = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.replace("\n", " "),
    "platform": platform.platform(),
    "cwd": os.getcwd(),
}

try:
    import torch
    probe["torch_status"] = "available"
    probe["torch_version"] = getattr(torch, "__version__", "unknown")
    cuda_available = bool(getattr(torch, "cuda", None) and torch.cuda.is_available())
    probe["cuda_available"] = cuda_available
    probe["cuda_device_count"] = torch.cuda.device_count() if cuda_available else 0
    probe["cuda_device_names"] = [torch.cuda.get_device_name(i) for i in range(probe["cuda_device_count"])] if cuda_available else []
except Exception as exc:
    probe.update({"torch_status": "unavailable", "torch_error": f"{type(exc).__name__}: {exc}", "cuda_available": False, "cuda_device_count": 0, "cuda_device_names": []})

try:
    import vllm
    probe["vllm_status"] = "available"
    probe["vllm_version"] = getattr(vllm, "__version__", "unknown")
except Exception as exc:
    probe["vllm_status"] = "unavailable"
    probe["vllm_error"] = f"{type(exc).__name__}: {exc}"

for key, value in probe.items():
    print(f"{key}: {value}")

In [ ]:
# Repository anchor table.
repo_root = "/mnt/cfs_bj_mt/workspace/limengjie03/my_school/RL-Kernel"
anchors = [
    {"path": f"{repo_root}/rl_engine/executors/vllm_sampler.py", "role": "vLLM shared-prefix sampler", "current_capability": "VLLMSamplerConfig carries num_generations, enable_prefix_caching, sampling_params, engine_kwargs, backend='vllm'; VLLMSharedPrefixSampler expands prompts once, calls vLLM, regroups outputs, and passes enable_prefix_caching to vllm.LLM.", "limitation": "Not proof of draft/target speculative decoding or measured speedup.", "issue_83_mapping": "Primary WS3 anchor for rollout integration and prefix-cache on/off validation."},
    {"path": f"{repo_root}/rl_engine/executors/rollout.py", "role": "Rollout executor and weight-update lifecycle", "current_capability": "generate_candidates lazily prepares VLLMSharedPrefixSampler; WeightUpdateManifest install/ack/reject/release lifecycle exists.", "limitation": "execute_rollout remains placeholder-style; not a finished optimized rollout runtime.", "issue_83_mapping": "WS3 parity checks for grouping, backend metadata, and weight-version readiness."},
    {"path": f"{repo_root}/tests/test_vllm_rollout_sampler.py", "role": "CPU-safe sampler/rollout tests", "current_capability": "Covers lazy vLLM import, prefix-cache flag propagation, grouped candidates, normalized output shape, rollout config validation, and weight lifecycle.", "limitation": "No A100 throughput or distributed TP measurement.", "issue_83_mapping": "Best existing CPU-safe regression base."},
    {"path": f"{repo_root}/rl_engine/kernels/ops/cuda/attention/prefix_shared_attn.py", "role": "Prefix-shared attention operator/fallback anchor", "current_capability": "Provides a prefix-shared attention operator/fallback anchor relevant to KV reuse and batch-invariance themes.", "limitation": "Not wired to vLLM PagedAttention KV management.", "issue_83_mapping": "P0.3/WS1 KV-cache consistency and batch-invariance anchor."},
    {"path": f"{repo_root}/benchmarks/benchmark_attention.py", "role": "Prefix-shared attention benchmark reference", "current_capability": "Benchmarks PrefixSharedAttentionOp vs expanded SDPA for GRPO-like shapes.", "limitation": "Operator-level only; no vLLM PagedAttention or speculative acceptance accounting.", "issue_83_mapping": "Conceptual validation reference, not rollout evidence."},
    {"path": f"{repo_root}/benchmarks/profiler.py", "role": "Profiling/report schema foundation", "current_capability": "Current workloads are logp-native, logp-fused, and sampling-native; CPU fallback/blocked metrics and CUDA latency/tokens/sec/TFLOPS/VRAM/backend/device metadata exist.", "limitation": "No vllm-rollout-prefix-cache workload in WORKLOAD_REGISTRY today.", "issue_83_mapping": "Schema/status inspiration for a future rollout profiler."},
    {"path": f"{repo_root}/scripts/run_profile_suite.py", "role": "Existing profiler CLI wrapper", "current_capability": "Runs operator-level profile suites and reports.", "limitation": "Does not validate vLLM rollout or prefix-cache performance.", "issue_83_mapping": "Operator-level smoke command only."},
    {"path": f"{repo_root}/benchmarks/benchmark_sampling.py", "role": "Sampling benchmark reference", "current_capability": "Compares native PyTorch sampling against SamplerBackend.", "limitation": "Single-process/operator-level, not multi-GPU rollout-level.", "issue_83_mapping": "Sampling behavior reference."},
    {"path": f"{repo_root}/site/getting_started/nvidia-a100-validation/a100_benchmark_notes.ipynb and {repo_root}/local_workspace/a100-smoke/a100_benchmark_notes.ipynb", "role": "Prior A100 smoke notebooks", "current_capability": "Existing one-GPU A100 smoke story, if present in the checkout.", "limitation": "Not issue #83 ranking, shared-prefix rollout design, or 8-GPU validation matrix.", "issue_83_mapping": "Differentiate this notebook from one-GPU smoke results."},
]
try:
    import pandas as pd
    display(pd.DataFrame(anchors))
except Exception:
    for row in anchors:
        print(row)

In [ ]:
# Issue #83 TODO inventory and ranking dataset.
# Planning found only two literal code TODOs:
# 1) rl_engine/kernels/sampling.py: AITER sampling operator TODO, low fit for NVIDIA A100.
# 2) rl_engine/kernels/ops/triton/loss/grpo_loss.py: larger-group tiled reduction TODO,
#    useful for GRPO scaling but less direct than vLLM shared-prefix rollout validation.
issue_83_todos = [
    {"todo_id": "WS3", "cluster": "vLLM rollout integration", "title": "Input parity, sampling/logprob decoupling, grouped candidates, prefix-cache metadata, and weight/version parity", "speculative_rollout_relevance": 5, "a100_fit": 5, "repo_anchor_coverage": 5, "validation_clarity": 5, "implementation_risk": 2, "done_signal": "CPU tests preserve grouping/metadata; guarded vLLM smoke validates prefix-cache on/off and num_generations when model/GPU exist."},
    {"todo_id": "P0.3/WS1", "cluster": "KV-cache path consistency", "title": "Batch-invariant attention/logprob and shared-prefix cache behavior checks", "speculative_rollout_relevance": 5, "a100_fit": 5, "repo_anchor_coverage": 4, "validation_clarity": 4, "implementation_risk": 3, "done_signal": "Prefix-cache on/off rows report grouping parity, logprob availability, and drift fields without PagedAttention claims."},
    {"todo_id": "WS4", "cluster": "Benchmark/report reproducibility", "title": "Reproducible matrix, result schema, status rules, and drift visualization readiness", "speculative_rollout_relevance": 4, "a100_fit": 5, "repo_anchor_coverage": 4, "validation_clarity": 5, "implementation_risk": 2, "done_signal": "Notebook emits a concrete matrix, measured-vs-planned schema, and blocked/fallback/pass rules."},
    {"todo_id": "P0.3/WS2", "cluster": "Distributed chain validation", "title": "TP/SP consistency, RNG/source alignment, and 1/2/4/8 GPU scaling readiness", "speculative_rollout_relevance": 4, "a100_fit": 5, "repo_anchor_coverage": 3, "validation_clarity": 4, "implementation_risk": 3, "done_signal": "TP templates and matrix rows define 1/2/4/8 GPU comparisons; 8-GPU rows block unless hardware exists."},
    {"todo_id": "P1", "cluster": "vLLM PagedAttention KV integration", "title": "Wire prefix_shared_attention concepts into vLLM PagedAttention KV management", "speculative_rollout_relevance": 5, "a100_fit": 5, "repo_anchor_coverage": 2, "validation_clarity": 2, "implementation_risk": 5, "done_signal": "Future-only: requires vLLM internals work and measured validation before claims."},
    {"todo_id": "P3", "cluster": "Kernel experiments", "title": "MLA-aware prefix-shared rollout kernels, fused sampler kernels, and other high-risk CUDA/vLLM internals", "speculative_rollout_relevance": 3, "a100_fit": 4, "repo_anchor_coverage": 2, "validation_clarity": 2, "implementation_risk": 5, "done_signal": "Deferred until rollout path and metrics prove the need."},
]
try:
    import pandas as pd
    display(pd.DataFrame(issue_83_todos))
except Exception:
    for item in issue_83_todos:
        print(item)

In [ ]:
# Transparent ranking function.
def score_todo(item):
    return 3 * item["speculative_rollout_relevance"] + 2 * item["a100_fit"] + 2 * item["repo_anchor_coverage"] + item["validation_clarity"] - 2 * item["implementation_risk"]

ranked_todos = sorted([dict(item, score=score_todo(item)) for item in issue_83_todos], key=lambda row: row["score"], reverse=True)
expected_top_order = ["WS3", "P0.3/WS1", "WS4", "P0.3/WS2", "P1", "P3"]
actual_top_order = [row["todo_id"] for row in ranked_todos]
assert actual_top_order == expected_top_order, (actual_top_order, expected_top_order)

for rank, row in enumerate(ranked_todos, 1):
    print(f"{rank}. {row['todo_id']} ({row['score']}): {row['title']}")
try:
    import pandas as pd
    display(pd.DataFrame(ranked_todos)[["todo_id", "cluster", "score", "title", "done_signal"]])
except Exception:
    pass

## Selected contribution explanation

The selected contribution is a **shared-prefix, multi-candidate rollout validation harness** centered on the current vLLM rollout path.

Current vLLM shared-prefix rollout is speculative-ready/multi-candidate rollout, not confirmed draft/target speculative decoding. Acceptance accounting fields are included only as schema placeholders until a true draft/target path exists. This lane creates the missing bridge between repository anchors and an 8xA100 validation plan while avoiding risky one-stage changes to vLLM internals, CUDA kernels, or benchmark registries.

In [ ]:
# Pure-Python experiment matrix generator.
from itertools import product

def derive_tier(num_gpus, prompt_count, prompt_length_bucket, completion_length, num_generations):
    if num_gpus in (1, 2) and prompt_count <= 8 and prompt_length_bucket <= 128 and completion_length <= 64 and num_generations <= 4:
        return "smoke"
    if num_gpus == 8 and (prompt_count >= 128 or prompt_length_bucket >= 2048 or num_generations >= 16):
        return "stress"
    return "scaling"

matrix = []
for vals in product([1, 2, 4, 8], [1, 2, 4, 8], ["off", "on"], [8, 32, 128], ["short_shared_prefix", "long_shared_prefix", "mixed_prefix"], [128, 512, 2048], [64, 256], [2, 4, 8, 16], [0.0, 0.7, 1.0], [0.9, 0.95], [50], ["na_current_rollout", "target_0.3_future", "target_0.5_future", "target_0.7_future", "target_0.9_future"]):
    num_gpus, tp_size, prefix_cache, prompt_count, prompt_set, prompt_length_bucket, completion_length, num_generations, temperature, top_p, top_k, acceptance_scenario_planned = vals
    if tp_size > num_gpus:
        continue
    candidate_count = prompt_count * num_generations
    matrix.append({"num_gpus": num_gpus, "tp_size": tp_size, "prefix_cache": prefix_cache, "prompt_count": prompt_count, "prompt_set": prompt_set, "prompt_length_bucket": prompt_length_bucket, "completion_length": completion_length, "num_generations": num_generations, "temperature": temperature, "top_p": top_p, "top_k": top_k, "acceptance_scenario_planned": acceptance_scenario_planned, "candidate_count": candidate_count, "requested_completion_tokens": candidate_count * completion_length, "tier": derive_tier(num_gpus, prompt_count, prompt_length_bucket, completion_length, num_generations)})

assert matrix, "experiment matrix must not be empty"
assert all(row["tp_size"] <= row["num_gpus"] for row in matrix)
assert all(row["candidate_count"] == row["prompt_count"] * row["num_generations"] for row in matrix)
assert {"smoke", "scaling", "stress"}.issubset({row["tier"] for row in matrix})
assert any(row["num_gpus"] == 8 for row in matrix)

print(f"matrix_rows: {len(matrix)}")
print(f"smoke_rows: {sum(1 for row in matrix if row['tier'] == 'smoke')}")
print(f"scaling_rows: {sum(1 for row in matrix if row['tier'] == 'scaling')}")
print(f"stress_rows: {sum(1 for row in matrix if row['tier'] == 'stress')}")
print("first_five_rows:")
for row in matrix[:5]:
    print(row)
try:
    import pandas as pd
    df_matrix = pd.DataFrame(matrix)
    display(df_matrix.head(10))
except Exception:
    df_matrix = None

In [ ]:
# Metrics schema: schema-only, not benchmark evidence.
measured_current_fields = ["run_id", "timestamp_utc", "gpu_count", "gpu_name", "gpu_architecture", "backend", "model", "tp_size", "prefix_cache_enabled", "num_prompts", "num_generations", "prompt_tokens_mean", "generated_tokens_total", "prompts_per_sec", "candidates_per_sec", "tokens_per_sec", "latency_ms_p50", "latency_ms_p95", "peak_vram_gb_per_gpu", "grouping_complete", "logprob_available", "logprob_max_abs_diff_or_na", "rng_seed", "weight_version", "status", "notes"]
future_planned_fields = ["draft_tokens_total_or_na", "accepted_tokens_total_or_na", "acceptance_rate_or_na", "draft_acceptance_target_planned"]
metrics_schema = {"measured_current_fields": measured_current_fields, "future_planned_speculative_accounting_fields": future_planned_fields, "status_values": ["pass", "blocked", "fallback", "not_applicable"], "schema_note": "Columns define future result records; placeholder values are not benchmark evidence."}
schema_only_example = {field: None for field in measured_current_fields + future_planned_fields}
schema_only_example.update({"run_id": "schema-example-not-a-result", "backend": "vllm", "gpu_architecture": "Ampere/A100 planned", "status": "blocked", "notes": "Example row only; not a measured run.", "draft_tokens_total_or_na": "not_applicable", "accepted_tokens_total_or_na": "not_applicable", "acceptance_rate_or_na": "not_applicable", "draft_acceptance_target_planned": "na_current_rollout"})
assert schema_only_example["run_id"].endswith("not-a-result")
assert schema_only_example["status"] in metrics_schema["status_values"]
assert schema_only_example["acceptance_rate_or_na"] == "not_applicable"
print(metrics_schema)
print(schema_only_example)

In [ ]:
# Validation gates and status rules.
status_rules = {
    "pass": "Required dependency and hardware were present, command ran, and validation checks passed.",
    "blocked": "Dependency, model, CUDA, vLLM, GPU count, or entrypoint was unavailable.",
    "fallback": "Command ran but used a fallback path rather than the intended optimized/backend path.",
    "not_applicable": "Planned/future field does not apply to current shared-prefix rollout.",
}
blocked_fallback_examples = [
    {"condition": "Missing CUDA", "status_for_a100_runs": "blocked", "status_for_pure_python_cells": "pass"},
    {"condition": "Missing vLLM", "status_for_rollout_templates": "blocked", "status_for_ranking_matrix_schema": "pass"},
    {"condition": "Missing model weights", "status_for_generation_templates": "blocked"},
    {"condition": "Fewer than 8 GPUs", "status_for_smoke_rows": "pass", "status_for_8xa100_rows": "blocked"},
    {"condition": "Existing profiler run", "status": "operator-level only; not evidence of vLLM rollout or speculative decoding performance"},
]
assert set(status_rules) == {"pass", "blocked", "fallback", "not_applicable"}
assert any(example["condition"] == "Fewer than 8 GPUs" for example in blocked_fallback_examples)
for key, value in status_rules.items():
    print(f"{key}: {value}")
for example in blocked_fallback_examples:
    print(example)

## CPU-safe validation commands

```bash
python -m json.tool local_workspace/issue-83-a100-speculative-rollout/issue_83_a100_speculative_rollout_plan.ipynb >/dev/null
python -m pytest tests/test_vllm_rollout_sampler.py -q
```

Optional broader CPU-safe command:

```bash
python -m pytest tests/test_vllm_rollout_sampler.py tests/test_profiler.py -q
```

These are validation commands for the implementation stage or later local checks. They do not require CUDA/vLLM/model weights when limited to JSON validation and CPU-safe tests. They do not claim A100 throughput, vLLM rollout performance, prefix-cache speedup, or speculative decoding speedup.

## Existing profiler smoke command: operator-level only

```bash
CUDA_VISIBLE_DEVICES=0 python scripts/run_profile_suite.py \
  --device cuda \
  --dtype float32 \
  --batch-sizes 2 \
  --seq-lens 16 \
  --vocab-sizes 1024 \
  --workloads logp-native,logp-fused,sampling-native \
  --warmup 0 \
  --repeat 1 \
  --output-dir reports/issue-83-a100-smoke \
  --csv \
  --json
```

This validates existing operator-level profiling/reporting paths only. It does **not** validate vLLM rollout, prefix-cache performance, PagedAttention KV integration, or speculative decoding.

## Manual vLLM RolloutExecutor smoke template

Requires vLLM and a valid local model path. Intended as a smoke probe, not a performance benchmark. It should be recorded as `blocked` if dependencies or weights are unavailable.

```bash
CUDA_VISIBLE_DEVICES=0 python - <<'PY'
from rl_engine.executors.rollout import RolloutExecutor

executor = RolloutExecutor({
    "model": "/path/to/local/model",
    "sampler": {
        "num_generations": 4,
        "enable_prefix_caching": True,
        "sampling_params": {
            "max_tokens": 64,
            "temperature": 0.7,
            "top_p": 0.9,
        },
    },
    "vllm": {
        "engine_kwargs": {
            "tensor_parallel_size": 1,
        },
    },
})
result = executor.generate_candidates([
    "Explain prefix cache validation in one paragraph."
])
print(result["backend"], result["prefix_cache_enabled"], result["num_prompts"], result["num_generations"])
print([[candidate.to_dict() for candidate in group] for group in result["normalized_outputs"]])
PY
```

## 2/4/8 GPU vLLM tensor-parallel templates

Adjust `/path/to/local/model` before running. These templates vary `CUDA_VISIBLE_DEVICES` and `tensor_parallel_size`. They are smoke templates, not benchmark evidence.

```bash
CUDA_VISIBLE_DEVICES=0,1 python - <<'PY'
from rl_engine.executors.rollout import RolloutExecutor
executor = RolloutExecutor({
    "model":"/path/to/local/model",
    "sampler":{"num_generations":4,"enable_prefix_caching":True,"sampling_params":{"max_tokens":64,"temperature":0.7,"top_p":0.9}},
    "vllm":{"engine_kwargs":{"tensor_parallel_size":2}},
})
print(executor.generate_candidates(["TP=2 prefix-cache smoke prompt."])["backend"])
PY
```

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3 python - <<'PY'
from rl_engine.executors.rollout import RolloutExecutor
executor = RolloutExecutor({
    "model":"/path/to/local/model",
    "sampler":{"num_generations":4,"enable_prefix_caching":True,"sampling_params":{"max_tokens":64,"temperature":0.7,"top_p":0.9}},
    "vllm":{"engine_kwargs":{"tensor_parallel_size":4}},
})
print(executor.generate_candidates(["TP=4 prefix-cache smoke prompt."])["backend"])
PY
```

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7 python - <<'PY'
from rl_engine.executors.rollout import RolloutExecutor
executor = RolloutExecutor({
    "model":"/path/to/local/model",
    "sampler":{"num_generations":4,"enable_prefix_caching":True,"sampling_params":{"max_tokens":64,"temperature":0.7,"top_p":0.9}},
    "vllm":{"engine_kwargs":{"tensor_parallel_size":8}},
})
print(executor.generate_candidates(["TP=8 prefix-cache smoke prompt."])["backend"])
PY
```

## Future distributed entrypoint placeholder

Future work only; this is not runnable until a future rollout validation entrypoint exists. Do not present it as current repo functionality.

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7 torchrun --nproc_per_node=8 \
  <future-rollout-validation-entrypoint> \
  --tp-size 8 \
  --prefix-cache on \
  --num-generations 8 \
  --report reports/issue-83-a100-speculative/8gpu.json
```

## Risks and mitigations

- **Risk: vLLM or model weights unavailable.** Mitigation: pure-Python ranking/matrix/schema cells run without vLLM; command cells are templates or guarded optional runs.
- **Risk: overclaiming speculative inference.** Mitigation: call this a speculative-rollout validation plan or speculative-ready rollout harness, not measured speculative decoding.
- **Risk: duplicating existing A100 smoke notebook.** Mitigation: focus on issue #83 TODO ranking, experiment design, vLLM shared-prefix rollout, and 8-GPU validation matrix rather than one-GPU smoke results.
- **Risk: kernel/vLLM internals work is too ambitious for one commit.** Mitigation: avoid modifying CUDA/vLLM internals in this stage; make the contribution a durable executable roadmap notebook.
- **Risk: acceptance metrics are mistaken for measured values.** Mitigation: mark acceptance-rate and draft-token fields as `or_na` / planned placeholders until a real draft/target speculative decoder is instrumented.

## Commit and validation checklist

1. Add only the notebook artifact for this stage:
   - `/mnt/cfs_bj_mt/workspace/limengjie03/my_school/RL-Kernel/local_workspace/issue-83-a100-speculative-rollout/issue_83_a100_speculative_rollout_plan.ipynb`
2. Validate notebook JSON:
   - `python -m json.tool /mnt/cfs_bj_mt/workspace/limengjie03/my_school/RL-Kernel/local_workspace/issue-83-a100-speculative-rollout/issue_83_a100_speculative_rollout_plan.ipynb >/dev/null`
3. Validate pure-Python notebook logic by executing or dry-running the ranking/matrix/schema/status cells.
4. Run CPU-safe test if feasible:
   - `python -m pytest /mnt/cfs_bj_mt/workspace/limengjie03/my_school/RL-Kernel/tests/test_vllm_rollout_sampler.py -q`
5. Do not require A100/vLLM/model weights for the notebook to be valid.
6. Do not commit generated benchmark reports unless explicitly requested later.
7. Do not modify runtime code, CUDA kernels, vLLM internals, or benchmark registries in this notebook-only stage.

## Final next steps and non-claims

Next steps are to use this matrix for CPU-safe tests first, optional vLLM smoke probes on available GPUs second, and controlled 1/2/4/8 GPU runs only when hardware and model weights are available. A future `vllm-rollout-prefix-cache` workload or acceptance-accounting path can reuse this schema, but neither is claimed as measured current functionality here.

This notebook does not claim measured 8xA100 results, PagedAttention KV integration, draft/target speculative decoding speedup, or a new CUDA kernel.